# On-going Assessment Lab01: Patient Readmission Prediction with Explainable AI

This notebook is prepared for **On-going Assessment Lab01**.

Project title: **Patient Readmission Prediction with Explainable AI**

The notebook follows the early project sessions in the syllabus table:

- Session 1--2: project topic introduction, group discussion, and topic registration
- Session 3--4: project planning and completion plan presentation
- Session 5: business understanding
- Session 6: data understanding and data collection
- Session 7: data preprocessing
- Session 8: importing datasets with Python
- Session 9: data wrangling

Because this is Lab01, the focus is not final deployment or advanced AI. The main goal is to understand the problem, import the data, clean the data, prepare the dataset, and create a clear plan for later machine learning and Explainable AI work.

## 1. Project Introduction

Patient readmission prediction is a healthcare machine learning problem. The main idea is to use hospital and patient information to estimate whether a patient may be readmitted after discharge.

This topic is useful because readmission can affect patient care quality, hospital workload, and resource planning. If a hospital can identify patients with higher readmission risk earlier, healthcare staff may plan follow-up care more carefully.

For this Lab01, the project only focuses on the early stages:

- Business understanding
- Data understanding
- Importing datasets
- Data wrangling
- Data preprocessing
- Basic exploratory data analysis
- Planning for later machine learning and Explainable AI

## 2. Business Understanding

The business problem is to prepare hospital data for a later model that predicts patient readmission. In simple academic language, this is a supervised learning problem if the dataset contains a readmission-related target column.

Healthcare context:

- Some patients may return to the hospital after discharge.
- Readmission may indicate that the patient needs better follow-up or care planning.
- A prediction model can support decision-making, but it should be explainable because healthcare decisions are sensitive.

Possible stakeholders:

- Hospital management
- Doctors
- Nurses
- Data analysts
- Patients

Expected business value:

- Reduce unnecessary readmission
- Improve patient care planning
- Support hospital resource allocation
- Help healthcare staff understand important risk factors

In this notebook, the target variable is detected programmatically. Possible target keywords include `readmitted`, `readmission`, `readmit`, `target`, and `outcome`.

In [ ]:
# Basic setup
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# Detect project root whether the notebook is opened from the root folder or notebooks folder
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Make sure src/ can be imported
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import find_csv_files, load_csv_files, download_with_kagglehub
from src.preprocessing import basic_preprocess, choose_prediction_dataset, detect_target_column
from src.utils import ensure_directories

ensure_directories(PROJECT_ROOT)

FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed_data"

print("Project root:", PROJECT_ROOT)
print("Figure output folder:", FIGURE_DIR)
print("Processed data folder:", PROCESSED_DIR)

## 3. Data Collection and Dataset Source

The dataset source is Kaggle:

`vanpatangan/readmission-dataset`

For this lab, three CSV files are uploaded manually to make code generation and testing easier. The notebook first searches for uploaded CSV files directly. Only if no CSV files are found, KaggleHub is used as a backup.

The code searches in:

- Current working directory
- `data/raw/`
- `/mnt/data/` in notebook environments

In [ ]:
# Search for uploaded CSV files first
search_folders = [
    PROJECT_ROOT,
    PROJECT_ROOT / "data" / "raw",
    Path("/mnt/data"),
]

csv_files = find_csv_files(search_folders, recursive=True)

if len(csv_files) == 0:
    print("No local CSV files found. Using KaggleHub as backup...")
    kaggle_path = download_with_kagglehub("vanpatangan/readmission-dataset")
    csv_files = find_csv_files([kaggle_path], recursive=True)
else:
    print("Local CSV files found. KaggleHub backup is not needed.")

print("\nCSV files detected:")
for file in csv_files:
    print("-", file)

## 4. Importing Dataset 1, Dataset 2, and Dataset 3

This section loads all available CSV files. Each dataset is shown separately so the team can understand the available files before choosing the main prediction dataset.

In [ ]:
# Load all detected CSV files
datasets = load_csv_files(csv_files)

print("Number of datasets loaded:", len(datasets))

for name, df in datasets.items():
    print("\n" + "=" * 80)
    print("Dataset name:", name)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head())

**Comment:** At this stage, the datasets are only imported and displayed. The next step is to inspect their structure, missing values, duplicates, and basic statistics.

## 5. Data Understanding

For each CSV file, this section checks:

- Shape
- Data types
- Missing values
- Duplicated rows
- Descriptive statistics
- Numeric and categorical columns

This matches the Lab01 focus on understanding the available data before modeling.

In [ ]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print("DATA UNDERSTANDING FOR:", name)
    print("=" * 80)

    print("\nShape:")
    print(df.shape)

    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))

    print("\nMissing values:")
    missing_table = df.isna().sum().to_frame("missing_count")
    missing_table["missing_percent"] = (missing_table["missing_count"] / len(df) * 100).round(2)
    display(missing_table)

    print("\nDuplicated rows:", df.duplicated().sum())

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    print("\nDescriptive statistics for numeric columns:")
    if len(numeric_cols) > 0:
        display(df[numeric_cols].describe().T)
    else:
        print("No numeric columns found.")

    print("\nDescriptive statistics for categorical columns:")
    if len(categorical_cols) > 0:
        display(df[categorical_cols].describe().T)
    else:
        print("No categorical columns found.")

    print("\nNumeric columns:", numeric_cols)
    print("Categorical columns:", categorical_cols)

**Comment:** Data understanding helps the group decide which dataset is suitable for prediction and what preprocessing steps are needed. Missing values and duplicates are checked before any model preview.

## 6. Data Wrangling

The wrangling process uses simple and transparent steps:

- Clean column names
- Remove duplicated rows
- Parse date columns if the column name suggests a date
- Fill missing numeric values with the median
- Fill missing categorical values with the mode or `Unknown`

The goal is not to over-clean the dataset, but to prepare it safely for Lab01 analysis.

In [ ]:
cleaned_datasets = {}

for name, df in datasets.items():
    cleaned_df = basic_preprocess(df)
    cleaned_datasets[name] = cleaned_df

    print("\n" + "=" * 80)
    print("Cleaned dataset:", name)
    print("Original shape:", df.shape)
    print("Cleaned shape:", cleaned_df.shape)
    print("Duplicated rows after cleaning:", cleaned_df.duplicated().sum())
    print("Columns after cleaning:", list(cleaned_df.columns))
    display(cleaned_df.head())

**Comment:** Column names are now easier to use in Python because they are lowercase and use underscores. Duplicate rows are removed, while missing values are handled with simple rules.

## 7. Data Preprocessing

This section chooses the most relevant dataset for prediction. The notebook looks for a target column related to readmission. If a target column exists, features and target are separated. If not, the notebook still saves a prepared dataset for future work.

Categorical variables are encoded using `pandas.get_dummies`, which is simple and suitable for Lab01.

In [ ]:
# Choose the most relevant dataset for prediction
selected_name, target_col = choose_prediction_dataset(cleaned_datasets)
main_df = cleaned_datasets[selected_name].copy()
original_main_df = datasets[selected_name].copy()

print("Selected dataset:", selected_name)
print("Selected dataset shape:", main_df.shape)
print("Detected target column:", target_col)

display(main_df.head())

if target_col is not None:
    print("\nTarget distribution:")
    display(main_df[target_col].value_counts(dropna=False).to_frame("count"))

    # Remove rows with missing target values, if any
    main_df = main_df.dropna(subset=[target_col]).copy()

    y = main_df[target_col]
    X = main_df.drop(columns=[target_col])

    # ID columns are usually not useful as predictive features
    id_like_cols = [col for col in X.columns if col == "id" or col.endswith("_id")]
    if len(id_like_cols) > 0:
        print("ID-like columns removed from features:", id_like_cols)
        X = X.drop(columns=id_like_cols)

    # Convert datetime columns to simple numeric values if any exist
    for col in X.select_dtypes(include="datetime").columns:
        X[col] = X[col].map(pd.Timestamp.toordinal)

    X_encoded = pd.get_dummies(X, drop_first=True)
    processed_df = pd.concat([X_encoded, y.reset_index(drop=True)], axis=1)
else:
    print("\nNo clear readmission target column was found.")
    print("Lab01 will focus on data preparation only.")

    X = main_df.copy()
    for col in X.select_dtypes(include="datetime").columns:
        X[col] = X[col].map(pd.Timestamp.toordinal)

    processed_df = pd.get_dummies(X, drop_first=True)

output_path = PROCESSED_DIR / "processed_readmission_lab01.csv"
processed_df.to_csv(output_path, index=False)

print("\nProcessed dataset shape:", processed_df.shape)
print("Processed dataset saved to:", output_path)
display(processed_df.head())

**Comment:** The processed dataset is saved for later machine learning work. If a readmission target column exists, the dataset is ready for a simple baseline model preview.

## 8. Exploratory Data Analysis

This section creates simple visualizations for Lab01. The visualizations are meant to support data understanding, not final model explanation.

Required charts included:

- Target variable distribution if available
- Missing value bar chart
- Numeric feature distributions
- Correlation heatmap for numeric columns
- Readmission-related comparison if target exists

In [ ]:
def save_and_show(filename):
    """Save the current figure and display it."""
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved figure:", path)

# Visualization 1: target distribution, or dataset row count if no target exists
if target_col is not None:
    target_counts = main_df[target_col].value_counts().sort_index()
    plt.figure(figsize=(7, 5))
    plt.bar(target_counts.index.astype(str), target_counts.values)
    plt.title("Target Variable Distribution")
    plt.xlabel(target_col)
    plt.ylabel("Number of patients")
    save_and_show("01_target_distribution.png")
else:
    dataset_sizes = pd.Series({name: df.shape[0] for name, df in cleaned_datasets.items()})
    plt.figure(figsize=(8, 5))
    plt.bar(dataset_sizes.index.astype(str), dataset_sizes.values)
    plt.title("Dataset Row Counts")
    plt.xlabel("Dataset")
    plt.ylabel("Number of rows")
    plt.xticks(rotation=30, ha="right")
    save_and_show("01_dataset_row_counts.png")

# Visualization 2: missing values in the selected raw dataset
missing_counts = original_main_df.isna().sum().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(missing_counts.index.astype(str), missing_counts.values)
plt.title("Missing Values by Column")
plt.xlabel("Column")
plt.ylabel("Missing count")
plt.xticks(rotation=45, ha="right")
save_and_show("02_missing_values.png")

# Select numeric feature columns for more visualizations
numeric_cols = main_df.select_dtypes(include="number").columns.tolist()
if target_col in numeric_cols:
    numeric_feature_cols = [col for col in numeric_cols if col != target_col]
else:
    numeric_feature_cols = numeric_cols.copy()

# Visualization 3 and 4: numeric distributions
for index, col in enumerate(numeric_feature_cols[:2], start=3):
    plt.figure(figsize=(8, 5))
    plt.hist(main_df[col].dropna(), bins=20)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    save_and_show(f"0{index}_{col}_distribution.png")

# Visualization 5: correlation heatmap using seaborn
corr_cols = numeric_feature_cols.copy()
if target_col is not None and target_col in numeric_cols:
    corr_cols.append(target_col)

if len(corr_cols) >= 2:
    plt.figure(figsize=(9, 6))
    corr_matrix = main_df[corr_cols].corr()
    sns.heatmap(corr_matrix, annot=True, fmt=".2f")
    plt.title("Correlation Heatmap for Numeric Columns")
    plt.xlabel("Numeric columns")
    plt.ylabel("Numeric columns")
    save_and_show("05_correlation_heatmap.png")
else:
    print("Not enough numeric columns for a correlation heatmap.")

# Visualization 6: readmission-related comparison if target exists
if target_col is not None and len(numeric_feature_cols) > 0:
    compare_col = numeric_feature_cols[0]
    plt.figure(figsize=(8, 5))
    main_df.boxplot(column=compare_col, by=target_col)
    plt.title(f"{compare_col} by {target_col}")
    plt.suptitle("")
    plt.xlabel(target_col)
    plt.ylabel(compare_col)
    save_and_show("06_readmission_comparison.png")

**Comment:** These charts give a first look at the data. More advanced visual analysis and explainability can be added in later project stages.

## 9. Baseline Machine Learning Preview

This section is only a small preview. It is included only if a clear target column exists.

The baseline model is intentionally simple because Lab01 should mainly focus on data understanding and preprocessing. Stronger models and explainable AI will be developed in later labs.

In [ ]:
if target_col is not None:
    from sklearn.model_selection import train_test_split
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

    X_model = X_encoded.copy()
    y_model = y.copy()

    if y_model.nunique() < 2:
        print("The target has fewer than 2 classes, so classification cannot be previewed.")
    else:
        stratify_value = y_model if y_model.value_counts().min() >= 2 else None

        X_train, X_test, y_train, y_test = train_test_split(
            X_model,
            y_model,
            test_size=0.2,
            random_state=42,
            stratify=stratify_value,
        )

        baseline_model = DecisionTreeClassifier(max_depth=4, random_state=42)
        baseline_model.fit(X_train, y_train)
        y_pred = baseline_model.predict(X_test)

        average_type = "binary" if set(pd.Series(y_model).dropna().unique()) <= {0, 1} else "weighted"

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average=average_type, zero_division=0)
        recall = recall_score(y_test, y_pred, average=average_type, zero_division=0)
        f1 = f1_score(y_test, y_pred, average=average_type, zero_division=0)

        print("Baseline model: Decision Tree Classifier")
        print("Accuracy:", round(accuracy, 4))
        print("Precision:", round(precision, 4))
        print("Recall:", round(recall, 4))
        print("F1-score:", round(f1, 4))

        cm = confusion_matrix(y_test, y_pred)
        cm_df = pd.DataFrame(cm)
        print("\nConfusion matrix:")
        display(cm_df)

        plt.figure(figsize=(6, 5))
        sns.heatmap(cm_df, annot=True, fmt="d")
        plt.title("Baseline Decision Tree Confusion Matrix")
        plt.xlabel("Predicted label")
        plt.ylabel("Actual label")
        save_and_show("07_baseline_confusion_matrix.png")
else:
    print("No target column was detected, so the baseline model preview is skipped.")

**Comment:** This baseline model is only a preview. The scores should not be treated as final results. Later labs should compare models, tune parameters, and evaluate the results more carefully.

## 10. Explainable AI Direction

Explainable AI is important in healthcare because predictions can affect patient care decisions. Healthcare staff should be able to understand why a model gives a certain risk prediction.

Possible future XAI methods:

- **Feature importance:** shows which variables are most influential in a model.
- **Permutation importance:** measures how much model performance changes when a feature is shuffled.
- **SHAP:** explains how each feature contributes to a prediction.
- **LIME:** explains individual predictions using a local interpretable model.

For Lab01, these methods are only introduced as future work. The current notebook prepares the data foundation for later modeling and explainability.

## 11. Project Planning

| Phase | Main Task | Expected Output |
|---|---|---|
| Phase 1 | Business understanding | Clear problem, stakeholders, and business value |
| Phase 2 | Data understanding | Dataset shape, columns, missing values, duplicates, statistics |
| Phase 3 | Data preprocessing | Cleaned and encoded dataset |
| Phase 4 | Baseline modeling | Simple classification model preview |
| Phase 5 | Model evaluation | Accuracy, precision, recall, F1-score, confusion matrix |
| Phase 6 | Explainable AI | Feature importance, SHAP, LIME, or permutation importance |
| Phase 7 | Final report and presentation | Final findings and project explanation |

This plan follows the early course sessions shown in the syllabus table. Lab01 mainly completes Phases 1--3 and gives only a small preview of Phase 4.

## 12. Lab01 Conclusion

In this Lab01 notebook:

- The patient readmission project was introduced.
- The business problem and stakeholders were defined.
- CSV datasets were imported automatically.
- Each dataset was checked for shape, columns, data types, missing values, duplicates, and statistics.
- Data wrangling and preprocessing were completed using simple rules.
- The likely readmission target column was detected if available.
- A processed dataset was saved for later work.
- Basic exploratory data analysis charts were created.
- A simple baseline model preview was included only when a valid target column existed.
- Explainable AI was introduced as a future direction.

The project is now ready for later machine learning and Explainable AI stages.